# Country‑Level Travel Safety Classifier

This model uses country level crime and resilience indicators to predict a 5 level travel safety class, ranging from very safe to very high risk, using a compact deep neural network trained on 2024 global crime data

## Environment Setup and Imports

Installs PyTorch and imports all required libraries, including pandas, NumPy, scikit‑learn, and PyTorch modules for building and training the neural network.

Sets a random seed and configures the device (CPU in your run) to ensure reproducible training runs

In [2]:
!pip install torch torchvision torchaudio

  Using cached torch-2.8.0-cp39-cp39-win_amd64.whl.metadata (30 kB)
  Using cached torchvision-0.23.0-cp39-cp39-win_amd64.whl.metadata (6.1 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached torch-2.8.0-cp39-cp39-win_amd64.whl (241.2 MB)
Using cached torchvision-0.23.0-cp39-cp39-win_amd64.whl (1.6 MB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 23.7 MB/s eta 0:00:00
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: C:\Users\ianmc\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
# 1. Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# 2. Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

## Data Loading and Column Standardization

Loads the 2024 “crime rate by country” dataset from CSV into global_df and prints the first few rows to confirm structure.

Renames the raw column names (e.g., crimeRateByCountry_crimeIndex) into cleaner, consistent names (crime_index, safety_index, etc.) so later code can reference them easily.

In [5]:
# 3. Load the cleaned crime/safety dataset
global_path = "data/crime-rate-by-country-2024.csv"
global_df = pd.read_csv(global_path)
global_df.head()

,country,crimeRateByCountry_crimeIndex,CrimeRate_OverallCriminalityScoreGOCI,CrimeRate_CriminalMarketsScore,CrimeRate_CriminalActorsScore,CrimeRate_ResilienceScore,CrimeRateSafetyIndex
0,India,44.4,5.75,6.70,4.8,5.42,55.6
1,China,60.8,6.37,6.53,6.2,5.67,39.2
2,United States,49.2,5.67,5.83,5.5,7.13,50.8
3,Indonesia,45.9,6.85,6.60,7.1,4.25,54.1
4,Pakistan,42.8,6.03,6.27,5.8,3.96,57.2


In [6]:
global_df.columns

Index(['country', 'crimeRateByCountry_crimeIndex',
       'CrimeRate_OverallCriminalityScoreGOCI',
       'CrimeRate_CriminalMarketsScore', 'CrimeRate_CriminalActorsScore',
       'CrimeRate_ResilienceScore', 'CrimeRateSafetyIndex'],
      dtype='object')

In [9]:
# Standardize column names from the raw ta_data table
global_df = global_df.rename(columns={
    "Country": "country",
    "crimeRateByCountry_crimeIndex": "crime_index",
    "CrimeRate_OverallCriminalityScoreGOCI": "overall_criminality_score",
    "CrimeRate_CriminalMarketsScore": "criminal_markets_score",
    "CrimeRate_CriminalActorsScore": "criminal_actors_score",
    "CrimeRate_ResilienceScore": "resilience_score",
    "CrimeRateSafetyIndex": "safety_index",
})

global_df.columns.tolist()

['country',
 'crime_index',
 'overall_criminality_score',
 'criminal_markets_score',
 'criminal_actors_score',
 'resilience_score',
 'safety_index',
 'country_norm']

## Data Cleaning 

Normalizes country names into a country_norm column and drops any rows missing core indices (crime_index, safety_index).

Identifies the main numeric features (crime index, safety index, overall criminality, markets, actors, resilience) and fills any remaining missing values with the median of each column, then summarizes their distributions.

In [10]:
# Normalize country names (useful later when joining with APIs)
global_df["country_norm"] = global_df["country"].str.strip().str.lower()

# Keep only rows with core indices present
core_cols = ["crime_index", "safety_index"]
global_df = global_df.dropna(subset=core_cols).copy()

# Identify numeric columns we’ll use
numeric_cols = [
    "crime_index",
    "safety_index",
    "overall_criminality_score",
    "criminal_markets_score",
    "criminal_actors_score",
    "resilience_score",
]

# Simple median imputation for missing values in numeric features
for col in numeric_cols:
    if global_df[col].isna().any():
        median_val = global_df[col].median()
        global_df[col] = global_df[col].fillna(median_val)

global_df[numeric_cols].describe()

,crime_index,safety_index,overall_criminality_score,criminal_markets_score,criminal_actors_score,resilience_score
count,140.000000,140.000000,140.000000,140.000000,140.000000,140.000000
mean,45.762857,54.237143,5.333357,5.203429,5.472143,5.020143
std,15.254943,15.254943,1.157298,1.129403,1.322907,1.626000
min,14.300000,17.900000,2.580000,1.670000,2.400000,1.500000
25%,32.750000,43.900000,4.555000,4.492500,4.600000,3.990000
50%,46.500000,53.500000,5.200000,5.170000,5.500000,5.130000
75%,56.100000,67.250000,6.205000,6.007500,6.525000,5.920000
max,82.100000,85.700000,8.150000,8.130000,8.600000,8.630000


## Risk Score Construction and Label Definition

Standardizes the crime index and an inverted safety index to create comparable z‑scores, then combines them into a single risk_score representing overall risk.

Converts the continuous risk_score into a 5‑level ordinal label risk_class using quantiles, yielding balanced counts across the five safety categories.

In [11]:
# Standardize crime_index
global_df["crime_z"] = (
    global_df["crime_index"] - global_df["crime_index"].mean()
) / global_df["crime_index"].std()

# Invert and standardize safety_index (lower safety = more risk)
safety_inv = global_df["safety_index"].max() - global_df["safety_index"]
global_df["safety_z_inv"] = (safety_inv - safety_inv.mean()) / safety_inv.std()

# Combined risk_score (tune weights if desired)
global_df["risk_score"] = (
    0.6 * global_df["crime_z"] + 0.4 * global_df["safety_z_inv"]
)

# Convert risk_score to 5 ordered classes via quantiles
quantiles = global_df["risk_score"].quantile([0.2, 0.4, 0.6, 0.8]).values

def to_risk_class(x, qs=quantiles):
    if x <= qs[0]:
        return 0   # very safe
    elif x <= qs[1]:
        return 1   # safe
    elif x <= qs[2]:
        return 2   # moderate
    elif x <= qs[3]:
        return 3   # risky
    else:
        return 4   # very risky

global_df["risk_class"] = global_df["risk_score"].apply(to_risk_class)
global_df["risk_class"].value_counts().sort_index()

0    29
1    27
2    28
3    28
4    28
Name: risk_class, dtype: int64

## Feature Selection, Train/Test Split, and Scaling

Selects six core features (crime_index, safety_index, overall_criminality_score, criminal_markets_score, criminal_actors_score, resilience_score) as inputs to the model.

Splits the data into stratified training and test sets (80/20) and standardizes features with StandardScaler, ensuring the model sees normalized values.

In [12]:
# Features for first version - OG data only
feature_cols = [
    "crime_index",
    "safety_index",
    "overall_criminality_score",
    "criminal_markets_score",
    "criminal_actors_score",
    "resilience_score",
]

X = global_df[feature_cols].values
y = global_df["risk_class"].values

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.shape, X_test_scaled.shape

((112, 6), (28, 6))

## Tensor Preparation and DataLoaders

Converts the scaled NumPy arrays into PyTorch tensors for both training and test sets.

Wraps these tensors in TensorDataset and DataLoader objects to handle batching and shuffling during training.

In [13]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_ds = TensorDataset(X_train_tensor, y_train_tensor)
test_ds = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

## Model Architecture: SafetyNet

Defines the SafetyNet class, a small fully‑connected neural network with two hidden layers (64 and 32 units), ReLU activations, and dropout for regularization.

Instantiates the model with 6 input features and 5 output classes, and sets up cross‑entropy loss with an Adam optimizer plus weight decay.

In [14]:
class SafetyNet(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        return self.net(x)

num_features = X_train_scaled.shape[1]
num_classes = len(np.unique(y_train))

model = SafetyNet(num_features, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(), lr=1e-3, weight_decay=1e-4
)

model

SafetyNet(
  (net): Sequential(
    (0): Linear(in_features=6, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=32, out_features=5, bias=True)
  )
)

## Training Loop and Accuracy Tracking

Trains the model for 200 epochs, looping over mini‑batches, computing cross‑entropy loss, backpropagating gradients, and updating weights.

Every 20 epochs, reports training and test accuracy and tracks the best test accuracy (currently ~ 92.9%), saving the best model weights to safety_model.pt.

In [15]:
def evaluate_accuracy(loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / total

In [16]:
n_epochs = 200
best_test_acc = 0.0

for epoch in range(1, n_epochs + 1):
    model.train()
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

    if epoch % 20 == 0:
        train_acc = evaluate_accuracy(train_loader)
        test_acc = evaluate_accuracy(test_loader)
        print(
            f"Epoch {epoch:3d}/{n_epochs} "
            f"- train_acc={train_acc:.3f} test_acc={test_acc:.3f}"
        )
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            torch.save(model.state_dict(), "safety_model.pt")

print("Best test accuracy:", best_test_acc)

Epoch  20/200 - train_acc=0.571 test_acc=0.571
Epoch  40/200 - train_acc=0.830 test_acc=0.786
Epoch  60/200 - train_acc=0.884 test_acc=0.857
Epoch  80/200 - train_acc=0.920 test_acc=0.929
Epoch 100/200 - train_acc=0.920 test_acc=0.929
Epoch 120/200 - train_acc=0.938 test_acc=0.929
Epoch 140/200 - train_acc=0.955 test_acc=0.929
Epoch 160/200 - train_acc=0.955 test_acc=0.929
Epoch 180/200 - train_acc=0.955 test_acc=0.929
Epoch 200/200 - train_acc=0.973 test_acc=0.929
Best test accuracy: 0.9285714285714286


## Evaluation: Confusion Matrix and Classification Report

Uses the trained model to generate predictions on the test set and computes a confusion matrix to see where the model confuses neighboring risk classes.

Prints a detailed classification report showing precision, recall, and F1 for each risk class, with macro and weighted averages around 0.93, indicating strong performance across classes

In [17]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
with torch.no_grad():
    logits_test = model(X_test_tensor.to(device))
    y_pred = logits_test.argmax(dim=1).cpu().numpy()

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=3))

Confusion matrix:
[[6 0 0 0 0]
 [1 4 0 0 0]
 [0 0 5 0 0]
 [0 0 0 5 1]
 [0 0 0 0 6]]

Classification report:
              precision    recall  f1-score   support

           0      0.857     1.000     0.923         6
           1      1.000     0.800     0.889         5
           2      1.000     1.000     1.000         5
           3      1.000     0.833     0.909         6
           4      0.857     1.000     0.923         6

    accuracy                          0.929        28
   macro avg      0.943     0.927     0.929        28
weighted avg      0.939     0.929     0.928        28



## Saved Model, Inference Helper and Testing Functions

Reloads the best model weights from disk and defines a helper function that takes a single country row, applies the scaler, runs the model, and returns the predicted class and probability vector

Demonstrates inference on an example country, mapping the predicted class to a human‑readable label such as “Moderate risk”.

Tests and displays random countries. 

In [18]:
# Reload model (for deployment / later use)
loaded_model = SafetyNet(num_features, num_classes)
loaded_model.load_state_dict(torch.load("safety_model.pt", map_location="cpu"))
loaded_model.eval()

def predict_country_risk(row):
    """
    row: a pandas Series from global_df for a single country
    returns: (predicted_class_int, probability_vector)
    """
    arr = row[feature_cols].values.astype(float).reshape(1, -1)
    arr_scaled = scaler.transform(arr)
    xt = torch.tensor(arr_scaled, dtype=torch.float32)
    with torch.no_grad():
        logits = loaded_model(xt)
        probs = torch.softmax(logits, dim=1).numpy()[0]
        pred_class = int(probs.argmax())
    return pred_class, probs

# Example: first country in the dataset
sample = global_df.iloc[0]
pred_class, probs = predict_country_risk(sample)
pred_class, probs

(2,
 array([0.04393895, 0.22576791, 0.5384257 , 0.1682969 , 0.02357056],
       dtype=float32))

In [19]:
risk_labels = {
    0: "Very safe",
    1: "Safe",
    2: "Moderate risk",
    3: "High risk",
    4: "Very high risk",
}
risk_labels[pred_class]

'Moderate risk'

In [20]:
def show_random_country_prediction(df, n_samples=3, random_state=42):
    """
    Randomly sample n_samples countries, print their feature values,
    and show the model's predicted safety class with probabilities.
    """
    rng = np.random.default_rng(random_state)
    indices = rng.choice(len(df), size=n_samples, replace=False)

    for i, idx in enumerate(indices, start=1):
        row = df.iloc[idx]
        country_name = row["country"]

        print(f"\n=== Sample {i}: {country_name} ===")
        print("Feature values used by the model:")
        for col in feature_cols:
            print(f"  {col}: {row[col]:.3f}")

        pred_class, probs = predict_country_risk(row)
        label = risk_labels[pred_class]

        print("\nModel prediction:")
        print(f"  Predicted risk class: {pred_class} ({label})")
        print("  Class probabilities:")
        for cls_idx, p in enumerate(probs):
            print(f"    {cls_idx} ({risk_labels[cls_idx]}): {p:.3f}")

In [21]:
show_random_country_prediction(global_df, n_samples=3, random_state=123)


=== Sample 1: Belarus ===
Feature values used by the model:
  crime_index: 51.400
  safety_index: 48.600
  overall_criminality_score: 5.870
  criminal_markets_score: 5.330
  criminal_actors_score: 6.400
  resilience_score: 3.250

Model prediction:
  Predicted risk class: 3 (High risk)
  Class probabilities:
    0 (Very safe): 0.001
    1 (Safe): 0.013
    2 (Moderate risk): 0.142
    3 (High risk): 0.699
    4 (Very high risk): 0.145

=== Sample 2: Singapore ===
Feature values used by the model:
  crime_index: 23.100
  safety_index: 76.900
  overall_criminality_score: 3.470
  criminal_markets_score: 3.930
  criminal_actors_score: 3.000
  resilience_score: 7.830

Model prediction:
  Predicted risk class: 0 (Very safe)
  Class probabilities:
    0 (Very safe): 0.891
    1 (Safe): 0.109
    2 (Moderate risk): 0.000
    3 (High risk): 0.000
    4 (Very high risk): 0.000

=== Sample 3: United States ===
Feature values used by the model:
  crime_index: 49.200
  safety_index: 50.800
  overal

In [25]:
show_random_country_predictions(global_df, n_samples=3, random_state=123)
show_random_country_predictions(global_df, n_samples=3, random_state=999)

NameError: name 'show_random_country_predictions' is not defined

## File Export Options 

In [22]:
import ipynbname

nb_path = ipynbname.path()
nb_name = nb_path.name 
nb_name

'Saftey_DL_model.ipynb'

In [23]:
import sys
from pathlib import Path


# HTML
!"{sys.executable}" -m nbconvert --to html "{nb_name}"

[NbConvertApp] Converting notebook Saftey_DL_model.ipynb to html
[NbConvertApp] Writing 337410 bytes to Saftey_DL_model.html


In [24]:
# PDF 
!"{sys.executable}" -m nbconvert --to pdf "{nb_name}"

[NbConvertApp] Converting notebook Saftey_DL_model.ipynb to pdf
[NbConvertApp] Writing 70751 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] Running bibtex 1 time: ['bibtex', 'notebook']
[NbConvertApp] WARNING | b had problems, most likely because there were no citations
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 76714 bytes to Saftey_DL_model.pdf
